# Breast Cancer Wisconsin Diagnostic — Logistic Regression

## Machine Learning Project 2: Binary Classification

This project uses the **Breast Cancer Wisconsin (Diagnostic)** dataset to predict whether a breast mass is **malignant (M)** or **benign (B)** from measurements computed from digitized images of fine needle aspirates (FNA).

Although the model used here is called **Logistic Regression**, the task is a **classification problem** because the target contains two classes: malignant and benign.

### Project objectives

- Inspect and understand the dataset.
- Clean irrelevant columns and prepare the target variable.
- Explore the class distribution and missing values.
- Split the data into training and testing sets.
- Standardize numerical features without leaking information from the test set.
- Train a Logistic Regression classifier.
- Evaluate the model using accuracy, precision, recall, F1-score, confusion matrix, and ROC-AUC.
- Interpret the results and document limitations.

**Dataset:** Breast Cancer Wisconsin (Diagnostic)  
**Observations:** 569  
**Predictor variables:** 30 numerical features  
**Target:** `diagnosis` (`M` = malignant, `B` = benign)


## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)


## 2. Load the Dataset

Place the downloaded dataset at:

`data/data.csv`

The original UCI/WDBC file commonly contains an `Unnamed: 32` column when saved as CSV. That column is removed during cleaning.


In [ ]:
DATA_PATH = "data/data.csv"

data_df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {data_df.shape}")
data_df.head()


## 3. Initial Data Inspection

In [ ]:
data_df.info()


In [ ]:
data_df.describe(include="all").T


### Check for missing values

In [ ]:
missing_values = data_df.isnull().sum().sort_values(ascending=False)
missing_values[missing_values > 0]


## 4. Data Cleaning

The `id` column identifies observations and does not represent a predictive characteristic of the cell nuclei, so it is excluded from modeling.

The `Unnamed: 32` column is an empty artifact found in the common CSV version of this dataset and is also removed when present.


In [ ]:
columns_to_drop = [col for col in ["Unnamed: 32", "id"] if col in data_df.columns]

data_df = data_df.drop(columns=columns_to_drop)

print("Dropped columns:", columns_to_drop)
print("New shape:", data_df.shape)


## 5. Encode the Target Variable

The original target is:

- `B` = benign
- `M` = malignant

For binary classification, the target is encoded as:

- `0` = benign
- `1` = malignant


In [ ]:
data_df["diagnosis"] = data_df["diagnosis"].map({"B": 0, "M": 1})

data_df["diagnosis"].value_counts()


## 6. Exploratory Data Analysis

In [ ]:
diagnosis_counts = data_df["diagnosis"].value_counts().sort_index()

diagnosis_counts.plot(kind="bar", figsize=(6, 4))
plt.xticks([0, 1], ["Benign (0)", "Malignant (1)"], rotation=0)
plt.ylabel("Number of observations")
plt.title("Diagnosis Distribution")
plt.show()


The dataset contains more benign observations than malignant observations, so class distribution should be considered when interpreting model performance. The train/test split below uses stratification to preserve approximately the same class proportions in both subsets.

In [ ]:
print(data_df["diagnosis"].value_counts(normalize=True).rename({0: "Benign", 1: "Malignant"}))


### Feature correlations

Several measurements are related because they describe different aspects of the same cell nuclei. A correlation matrix can help identify strong relationships among predictors.


In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(
    data_df.corr(numeric_only=True),
    cmap="coolwarm",
    center=0,
    linewidths=0.2
)
plt.title("Feature Correlation Matrix")
plt.show()


## 7. Define Features and Target

In [ ]:
X = data_df.drop(columns=["diagnosis"])
y = data_df["diagnosis"]

print("X shape:", X.shape)
print("y shape:", y.shape)


## 8. Train-Test Split

The data is split before feature scaling.

This order matters: the scaler must learn its mean and standard deviation from the training set only. Using the entire dataset before the split would allow information from the test set to influence preprocessing and create data leakage.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)


## 9. Feature Scaling

Logistic Regression can benefit from standardized predictors, particularly when variables have very different numerical ranges.

`StandardScaler` is fitted only on `X_train`, then applied to both training and test data.


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training data scaled:", X_train_scaled.shape)
print("Testing data scaled:", X_test_scaled.shape)


## 10. Train the Logistic Regression Model

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)

lr.fit(X_train_scaled, y_train)


## 11. Generate Predictions

In [ ]:
y_pred = lr.predict(X_test_scaled)
y_prob = lr.predict_proba(X_test_scaled)[:, 1]

predictions = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Predicted": y_pred,
    "Malignant_Probability": y_prob
})

predictions.head(10)


## 12. Model Evaluation

Because this is a binary classification problem, accuracy alone is not enough.

- **Precision:** among observations predicted as malignant, the proportion that were actually malignant.
- **Recall:** among malignant observations, the proportion correctly identified by the model.
- **F1-score:** harmonic mean of precision and recall.
- **ROC-AUC:** measures how well the model separates the two classes across classification thresholds.


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"],
    "Score": [accuracy, precision, recall, f1, roc_auc]
})

metrics


### Classification report

In [ ]:
print(classification_report(
    y_test,
    y_pred,
    target_names=["Benign", "Malignant"]
))


## 13. Confusion Matrix

The confusion matrix shows how many benign and malignant observations were classified correctly or incorrectly.


In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Benign", "Malignant"]
)

disp.plot()
plt.title("Logistic Regression — Confusion Matrix")
plt.show()


### ROC Curve

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("Logistic Regression — ROC Curve")
plt.show()


## 14. Model Coefficients

Logistic Regression assigns a coefficient to each standardized feature. The coefficient sign indicates the direction of its association with the model's predicted log-odds of the malignant class, while its magnitude reflects the contribution within this standardized model.

Coefficient magnitude should be interpreted carefully because correlated predictors can share information.


In [ ]:
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": lr.coef_[0],
    "Absolute_Coefficient": np.abs(lr.coef_[0])
}).sort_values("Absolute_Coefficient", ascending=False)

coefficients.head(15)


## 15. Key Findings

After running the notebook, use this section to record the actual results produced by the model.

Suggested points to discuss:

1. The model's accuracy on the held-out test set.
2. The malignant-class recall, since missed malignant cases are especially important in this application.
3. The F1-score and ROC-AUC as additional measures of classification performance.
4. Patterns visible in the confusion matrix.
5. Features with comparatively large standardized coefficients.
6. Limitations such as the dataset size, feature correlation, and the fact that this model should not be treated as a clinical diagnostic system.

> **Important:** The numerical results should be written after the notebook has been executed so that the GitHub documentation reflects the actual run.


## 16. Conclusion

This project demonstrates an end-to-end binary classification workflow using Logistic Regression on the Breast Cancer Wisconsin (Diagnostic) dataset.

The workflow covers data inspection, cleaning, target encoding, exploratory analysis, stratified train-test splitting, leakage-safe feature scaling, model training, and multiple evaluation metrics.

The project provides a foundation for a later comparison with models such as Decision Trees, Random Forest, Support Vector Machines, and K-Nearest Neighbors.


## 17. Reproducibility

The experiment uses:

- `random_state=42` for reproducible splitting and model initialization.
- `StandardScaler` fitted only on the training data.
- A fixed dataset path: `data/data.csv`.

For a stronger production workflow, the preprocessing and model can later be combined in a scikit-learn `Pipeline`.
